# Member 04 — Recommendation & Explainability Engine
Independent rule-based prototype. **Not trained ML.** Example priorities are illustrative; rules need expert review.

In [ ]:
import pandas as pd
from google.colab import files
print("Upload member04_recommendation_dataset.csv from this package")
files.upload()

In [ ]:
import pandas as pd
from IPython.display import display

rules = pd.read_csv("member04_recommendation_dataset.csv")
print("Rules loaded:", len(rules))
display(rules[["rule_id", "issue_category", "guideline_reference"]])


In [ ]:
SEVERITY = {"Low": 1, "Medium": 2, "High": 3}

def recommend(issue_category, severity="Medium", user_impact=3):
    if severity not in SEVERITY:
        raise ValueError("severity must be Low, Medium or High")
    if not isinstance(user_impact, (int, float)) or not 1 <= user_impact <= 5:
        raise ValueError("user_impact must be between 1 and 5")
    match = rules.loc[rules["issue_category"] == issue_category]
    if match.empty:
        return {"issue_category": issue_category, "status": "no_rule_found"}
    r = match.iloc[0]
    # Demonstration heuristic, NOT a validated expert ranking
    score = round(0.6 * (SEVERITY[severity] / 3) + 0.4 * (user_impact / 5), 3)
    return {
        "issue_category": issue_category,
        "severity_input": severity,
        "user_impact_input": user_impact,
        "priority_score_demo": score,
        "recommendation": r["recommended_solution"],
        "explanation": r["explanation"],
        "guideline_reference": r["guideline_reference"],
        "rule_validation_status": r["validation_status"]
    }

def prioritize(issue_list):
    return sorted([recommend(**x) for x in issue_list],
                  key=lambda x: x.get("priority_score_demo", -1), reverse=True)

print("ENGINE READY")


In [ ]:
demo_issues = [
    {"issue_category": "low_text_contrast", "severity": "High", "user_impact": 5},
    {"issue_category": "missing_form_labels", "severity": "High", "user_impact": 4},
    {"issue_category": "inconsistent_navigation", "severity": "Low", "user_impact": 2},
]
results = prioritize(demo_issues)
display(pd.DataFrame(results))
pd.DataFrame(results).to_csv("member04_demo_results.csv", index=False)
print("DEMO COMPLETED. Results saved to member04_demo_results.csv")


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
category = widgets.Dropdown(options=sorted(rules.issue_category.tolist()), description="Issue:")
severity = widgets.Dropdown(options=["Low", "Medium", "High"], value="Medium", description="Severity:")
impact = widgets.IntSlider(value=3, min=1, max=5, description="Impact:")
button = widgets.Button(description="Get Recommendation", button_style="success")
output = widgets.Output()
def on_click(_):
    with output:
        clear_output()
        display(pd.DataFrame([recommend(category.value, severity.value, impact.value)]).T)
button.on_click(on_click)
display(category, severity, impact, button, output)


## Limitations
- Severity and user impact are supplied manually in this demo.
- Priority weighting is illustrative and not validated.
- WCAG references are starting points; review the full success criteria and exceptions.
- Future integration: ingest member outputs and trained predictions.